## ETAPA 1: Processamento dos Sinais (.mat)
Nesta fase, o script realiza a leitura dos sinais brutos, filtragem, normalização e a extração das características morfológicas (SUT, Área e Largura 25%) que servirão de entrada para a rede neural.

> **Importante:** É obrigatório possuir os arquivos no formato `.mat` do dataset PulseDB/MIMIC.

> **Download:** [Acesse o repositório Rutgers Box](https://rutgers.app.box.com/s/sw3c51fr5oybz6mhqsphh5zg8ibxw800/folder/267219030116)

> **Nota**: Cada arquivo é volumoso. Para testes iniciais, recomenda-se baixar apenas uma unidade.

### Instruções:
1. No script de processamento, localize a seção de configurações e altere a variável de diretório para o caminho local onde os arquivos .mat foram extraídos.
2. Execute o script Python.
3. O algoritmo gerará um arquivo `.npz` contendo os dados prontos para o treinamento.

In [ ]:
# SCRIPT DE PROCESSAMENTO (MLP - 3 CARACTERÍSTICAS)
# COM REAMOSTRAGEM DE 125Hz PARA 400Hz

import os
import numpy as np
import h5py  # Biblioteca para ler arquivos .mat (que são HDF5 v7.3)
from scipy.signal import butter, filtfilt, find_peaks, resample  # Funções de filtro e reamostragem
from scipy.integrate import trapz  # Para calcular a integral (Área)
from scipy.ndimage import gaussian_filter1d  # Para o filtro gaussiano (smoothdata)
from sklearn.model_selection import train_test_split  # Para dividir treino/teste
from tqdm import tqdm  # Barra de progresso
import traceback

# --- CONFIGURAÇÕES ---
DIRETORIO = 'C:/Documents/PulseDB_MIMIC/'  # Caminho onde você guardou os arquivos .mat  
FREQUENCIA_AMOSTRAGEM_ORIGINAL = 125  # Frequência original dos dados
FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO = 400  # Frequência alvo (para o ESP32)
ARQUIVO_SAIDA = f'dados_processados_3_features_{FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO}Hz.npz'

# FUNÇÕES AUXILIARES
def ler_dados_janela_h5(h5_file, subj_wins_group, indice_janela):
    def read_item(dataset, index):
        if h5py.check_ref_dtype(dataset.dtype):
            try:
                ref = dataset[0, index]
                if isinstance(h5_file[ref], h5py.Dataset):
                    value = h5_file[ref][:].squeeze()
                    return value.item() if value.size == 1 else value
            except Exception:
                return None
        try:
            value = dataset[index] if dataset.ndim == 1 or dataset.shape[0] > 1 else dataset[0, index]
            return value.item() if hasattr(value, 'item') and value.size == 1 else value
        except Exception:
            return None

    ppg_bruto = read_item(subj_wins_group['PPG_Raw'], indice_janela)
    abp_bruto = read_item(subj_wins_group['ABP_Raw'], indice_janela)
    sbp_seg = read_item(subj_wins_group['SegSBP'], indice_janela) 
    dbp_seg = read_item(subj_wins_group['SegDBP'], indice_janela) 

    # Limpeza de dados (garante que sejam arrays numpy ou NaN)
    if ppg_bruto is None or not isinstance(ppg_bruto, np.ndarray) or ppg_bruto.ndim == 0:
        ppg_bruto = np.array([])
    if abp_bruto is None or not isinstance(abp_bruto, np.ndarray) or abp_bruto.ndim == 0:
        abp_bruto = np.array([])
    if sbp_seg is None or not isinstance(sbp_seg, (int, float, np.number)) or np.isnan(sbp_seg):
        sbp_seg = np.nan
    if dbp_seg is None or not isinstance(dbp_seg, (int, float, np.number)) or np.isnan(dbp_seg):
        dbp_seg = np.nan

    return ppg_bruto.flatten(), abp_bruto.flatten(), sbp_seg, dbp_seg


def segmentacao_adaptativa_py(sinal, fs):
    # Detecta picos sistólicos.
    # Usa um filtro de média móvel e 'find_peaks' do SciPy.
    # Distância mínima baseada na Frequência Cardíaca máxima (180 BPM)
    distancia_minima = np.floor((60 / 180) * fs)
    if len(sinal) < 3:
        return np.array([])

    # Equivalente a 'smoothdata(sinal, 'movmean', 3)' do MATLAB
    sinal_suavizado = np.convolve(sinal, np.ones(3)/3, mode='same')

    # Define limites adaptativos (altura e proeminência)
    sinal_range = np.max(sinal_suavizado) - np.min(sinal_suavizado)
    if sinal_range < 1e-6:
        return np.array([])
    
    altura_minima = np.mean(sinal_suavizado) + 0.05 * sinal_range
    prominencia_minima = 0.05 * sinal_range

    if altura_minima <= 0: altura_minima = 1e-6
    if prominencia_minima <= 0: prominencia_minima = 1e-6

    try:
        # Encontra os picos usando os limites
        picos, _ = find_peaks(sinal_suavizado, distance=distancia_minima, height=altura_minima, prominence=prominencia_minima)
    except ValueError:
        picos = np.array([])

    # Pós-processamento para remover picos falsos muito próximos (ruído)
    if len(picos) > 1:
        picos_validos = [picos[0]]
        for i in range(1, len(picos)):
            if (picos[i] - picos_validos[-1]) / fs > 0.1:  # Intervalo mínimo de 0.1s
                picos_validos.append(picos[i])
        picos = np.array(picos_validos)

    return picos


def extrair_caracteristicas_3_features_py(segmento_ppg, fs, inicio, pico):
    # Extrai as 3 características morfológicas de um único batimento (segmento).
    num_features = 3
    caracteristicas = np.zeros(num_features)
    try:
        # --- Característica 1: Tempo de Subida (SUT) ---
        tempo_subida = (pico - inicio) / fs

        # --- Característica 2: Área do Pulso ---
        eixo_tempo = np.arange(0, len(segmento_ppg)) / fs
        area_pulso = trapz(segmento_ppg, eixo_tempo)

        # --- Característica 3: Largura a 25% da Amplitude ---
        segmento_suavizado = gaussian_filter1d(segmento_ppg, sigma=1.0)
        indice_pico_rel = pico - inicio

        # Validações
        if indice_pico_rel < 0 or indice_pico_rel >= len(segmento_suavizado):
            return np.zeros(num_features)

        # Amplitude do pulso (do vale ao pico)
        altura_pulso = segmento_suavizado[indice_pico_rel] - segmento_suavizado[0]
        if altura_pulso <= 1e-6:
            return np.zeros(num_features)

        # Define o limiar de 25%
        limiar_25 = segmento_suavizado[0] + 0.25 * altura_pulso

        # Encontra o primeiro e o último índice onde o sinal está acima do limiar
        indices_acima = np.where(segmento_suavizado >= limiar_25)[0]
        if len(indices_acima) < 2:
            return np.zeros(num_features)

        # Calcula a largura (em segundos)
        largura_amostras = indices_acima[-1] - indices_acima[0]
        largura_25 = largura_amostras / fs

        # Retorna o vetor de características
        caracteristicas = np.array([area_pulso, tempo_subida, largura_25])

        if np.any(np.isnan(caracteristicas)) or np.any(np.isinf(caracteristicas)):
            return np.zeros(num_features)

        return caracteristicas

    except Exception:
        return np.zeros(num_features)


# LÓGICA PRINCIPAL DE PROCESSAMENTO
print(f'--- Iniciando Processamento (3 Características) @ {FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO}Hz ---')
lista_arquivos = [f for f in os.listdir(DIRETORIO) if f.endswith('.mat') and not f.startswith('dados_processados')]

dados_por_paciente = []  # Lista para armazenar dados por paciente
b_ppg, a_ppg = butter(2, [0.5, 8], btype='band', fs=FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO)
b_abp, a_abp = butter(2, 10, btype='low', fs=FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO)

arquivos_lidos_ok = 0
batimentos_extraidos_total = 0

for id_paciente, nome_arquivo in enumerate(tqdm(lista_arquivos, desc="Processando Pacientes")):
    caminho_arquivo = os.path.join(DIRETORIO, nome_arquivo)
    arquivo_processado_com_sucesso = False

    try:
        with h5py.File(caminho_arquivo, 'r') as dados_h5:
            if 'Subj_Wins' not in dados_h5 or not all(k in dados_h5['Subj_Wins'] for k in ['PPG_Raw', 'ABP_Raw', 'SegSBP', 'SegDBP']):
                continue
            
            subj_wins_group = dados_h5['Subj_Wins']
            try:
                num_janelas = subj_wins_group['SegSBP'].shape[1]
            except Exception:
                continue

            caracteristicas_paciente, pressoes_paciente = [], []
            max_janelas = min(20, num_janelas)

            for i in range(max_janelas):
                ppg_bruto, abp_bruto, sbp_seg, dbp_seg = ler_dados_janela_h5(dados_h5, subj_wins_group, i)
                
                if np.isnan(sbp_seg) or np.isnan(dbp_seg): continue
                if ppg_bruto.size < 2 or abp_bruto.size < 2: continue

                # ETAPA 1: REAMOSTRAGEM
                num_amostras_original = len(ppg_bruto)
                num_amostras_nova = int(np.round(num_amostras_original * (FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO / FREQUENCIA_AMOSTRAGEM_ORIGINAL)))
                
                if num_amostras_nova < 2: continue
                try:
                    ppg_bruto_resampled = resample(ppg_bruto, num_amostras_nova)
                    abp_bruto_resampled = resample(abp_bruto, num_amostras_nova)
                except ValueError: continue 

                if len(ppg_bruto_resampled) < FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO * 2: continue

                # ETAPA 2: FILTRAGEM
                try:
                    ppg_filtrado = filtfilt(b_ppg, a_ppg, ppg_bruto_resampled)
                    abp_filtrado = filtfilt(b_abp, a_abp, abp_bruto_resampled)
                except ValueError: continue

                # ETAPA 3: NORMALIZAÇÃO (MIN-MAX 0-1)
                min_ppg_f, max_ppg_f = np.min(ppg_filtrado), np.max(ppg_filtrado)
                if max_ppg_f - min_ppg_f < 1e-6: continue
                ppg_normalizado = (ppg_filtrado - min_ppg_f) / (max_ppg_f - min_ppg_f)

                min_abp_f, max_abp_f = np.min(abp_filtrado), np.max(abp_filtrado)
                if max_abp_f - min_abp_f < 1e-6: continue
                abp_ajustado = dbp_seg + (abp_filtrado - min_abp_f) * (sbp_seg - dbp_seg) / (max_abp_f - min_abp_f)

                # ETAPA 4: SEGMENTAÇÃO (VALE-A-VALE)
                picos_ppg = segmentacao_adaptativa_py(ppg_normalizado, FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO)
                if len(picos_ppg) < 2: continue

                vales = [0]
                valid_picos = [p for p in picos_ppg if p+1 < len(ppg_normalizado)]
                if len(valid_picos) < 2: continue
                picos_ppg = np.array(valid_picos)

                for j in range(len(picos_ppg) - 1):
                    start_idx = picos_ppg[j]
                    end_idx = picos_ppg[j+1]
                    if start_idx < end_idx:
                        try:
                            vales.append(start_idx + np.argmin(ppg_normalizado[start_idx:end_idx]))
                        except ValueError: pass

                vales.append(len(ppg_normalizado)-1)
                vales = sorted(list(set(vales)))

                # ETAPA 5: EXTRAÇÃO DE CARACTERÍSTICAS
                for batimento in range(len(picos_ppg)):
                    pico = picos_ppg[batimento]
                    vales_antes = [v for v in vales if v < pico]
                    vales_depois = [v for v in vales if v > pico]
                    if not vales_antes or not vales_depois: continue
                    inicio, fim = max(vales_antes), min(vales_depois)

                    if not (inicio < pico < fim) or fim >= len(ppg_normalizado) or inicio < 0: continue
                    segmento_atual = ppg_normalizado[inicio:fim+1]
                    if len(segmento_atual) < 3: continue

                    caracteristicas = extrair_caracteristicas_3_features_py(segmento_atual, FREQUENCIA_AMOSTRAGEM_PROCESSAMENTO, inicio, pico)

                    if caracteristicas.sum() != 0 and not np.any(np.isnan(caracteristicas)):
                        if inicio < len(abp_ajustado) and fim < len(abp_ajustado):
                            segmento_abp = abp_ajustado[inicio:fim+1]
                            if len(segmento_abp) > 0:
                                sbp, dbp = np.max(segmento_abp), np.min(segmento_abp)
                                if 70 < sbp < 200 and 40 < dbp < 120:
                                    caracteristicas_paciente.append(caracteristicas)
                                    pressoes_paciente.append([sbp, dbp])
                                    batimentos_extraidos_total += 1

            if caracteristicas_paciente:
                arquivo_processado_com_sucesso = True
                dados_por_paciente.append({
                    "id": id_paciente, 
                    "caracteristicas": np.array(caracteristicas_paciente), 
                    "pressoes": np.array(pressoes_paciente)
                })
                arquivos_lidos_ok += 1

    except Exception as e:
        if not (isinstance(e, OSError) and 'Unable to open file' in str(e)):
            print(f"\nErro no arquivo {nome_arquivo}: {e}.")

print("\n--- Resumo do Processamento (3 Features) ---")
print(f"Arquivos Tentados: {len(lista_arquivos)}")
print(f"Arquivos que Produziram Dados: {arquivos_lidos_ok}")
print(f"Total de Batimentos Válidos Extraídos: {batimentos_extraidos_total}")
print(f"Total de Pacientes com Dados Salvos: {len(dados_por_paciente)}")
print("-------------------------------\n")

if not dados_por_paciente or batimentos_extraidos_total == 0:
    print("\nERRO CRÍTICO: Nenhum dado válido foi extraído após processamento.")
else:
    ids_unicos = np.array([d["id"] for d in dados_por_paciente])

    def agregar_dados(dados_lista, ids_selecionados):
        if len(ids_selecionados) == 0: return np.array([]).reshape(0,3), np.array([]).reshape(0,2)
        indices = [i for i, d in enumerate(dados_lista) if d["id"] in ids_selecionados]
        if not indices: return np.array([]).reshape(0,3), np.array([]).reshape(0,2)
        
        lista_caracteristicas = [dados_lista[i]["caracteristicas"] for i in indices if dados_lista[i]["caracteristicas"].size > 0]
        lista_pressoes = [dados_lista[i]["pressoes"] for i in indices if dados_lista[i]["pressoes"].size > 0]
        
        if not lista_caracteristicas or not lista_pressoes: 
            return np.array([]).reshape(0,3), np.array([]).reshape(0,2)
        
        return np.vstack(lista_caracteristicas), np.vstack(lista_pressoes)

    if len(ids_unicos) < 3:
        print("AVISO: Número de pacientes baixo para divisão ideal.")
        test_size = 0.15 if len(ids_unicos) > 1 else 0
        val_split = 0.15 / (1.0 - test_size) if (1.0 - test_size) > 0 else 0
        if len(ids_unicos) <= 1: 
            train_val_ids, test_ids = ids_unicos, np.array([])
        else: 
            train_val_ids, test_ids = train_test_split(ids_unicos, test_size=test_size, random_state=42, shuffle=True)
        
        if len(train_val_ids) <= 1: 
            train_ids, val_ids = train_val_ids, np.array([])
        else: 
            train_ids, val_ids = train_test_split(train_val_ids, test_size=val_split, random_state=42, shuffle=True)
    else:
        train_val_ids, test_ids = train_test_split(ids_unicos, test_size=0.15, random_state=42, shuffle=True)
        train_ids, val_ids = train_test_split(train_val_ids, test_size=(0.15/0.85), random_state=42, shuffle=True)

    print(f"IDs para Agregação -> Treino: {len(train_ids)}, Validação: {len(val_ids)}, Teste: {len(test_ids)}")

    X_train, y_train = agregar_dados(dados_por_paciente, train_ids)
    X_val, y_val = agregar_dados(dados_por_paciente, val_ids)
    X_test, y_test = agregar_dados(dados_por_paciente, test_ids)

    if X_train.size == 0 and X_val.size == 0 and X_test.size == 0:
        print("\nERRO FINAL: Nenhum dado agregado para salvar.")
    else:
        # Garante shapes corretos
        X_train = X_train.reshape(-1, 3)
        y_train = y_train.reshape(-1, 2)
        X_val = X_val.reshape(-1, 3)
        y_val = y_val.reshape(-1, 2)
        X_test = X_test.reshape(-1, 3)
        y_test = y_test.reshape(-1, 2)

        np.savez_compressed(ARQUIVO_SAIDA,
                            X_train=X_train, y_train=y_train,
                            X_val=X_val, y_val=y_val,
                            X_test=X_test, y_test=y_test)
        
        print(f'\nProcessamento concluído! Arquivo salvo: {ARQUIVO_SAIDA}')
        print(f'Total de amostras salvas: Treino={len(X_train)}, Validação={len(X_val)}, Teste={len(X_test)}')

--- Iniciando Processamento (3 Características) @ 400Hz ---


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Documents/PulseDB_MIMIC/'

## ETAPA 2: Treinamento da Rede Neural (MLP)

Utilizamos os dados processados para treinar uma rede neural MLP capaz de prever a Pressão Arterial Sistólica (SBP) e Diastólica (DBP).

### O que este script faz:
*   **Normalização Z-Score:** Ajusta as características para média 0 e desvio 1.
*   **Arquitetura:** Camadas Densas com ativação ReLU e Dropout para evitar overfitting.
*   **Saída:** Exporta o modelo treinado e os parâmetros do Scaler (essenciais para o ESP32).

### Instruções:
1. Garanta que o arquivo `.npz` da Etapa 1 está na mesma pasta.
2. Execute o script de treinamento.
3. Verifique os resultados de erro (MAE) no console ao final da execução.

### Arquivos Gerados:
* ```ann_MLP_3_features_400Hz.keras```: O modelo treinado e pronto para uso.

* ```scaler_3_features_400Hz.npz```: Contém a média e o desvio padrão das características. 
  **Guarde este arquivo**(ou apenas guarde os 3 valores da média e os 3 valores do desvio que aparecerão na saída, após a execução do script), pois ele é essencial para a predição em tempo real no microcontrolador.

In [ ]:
# SCRIPT DE TREINAMENTO DA MLP (3 CARACTERÍSTICAS @ 400Hz)

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler  # Para Normalização Z-Score
import os

print("--- Iniciando Treinamento da MLP (3 Caracteristicas @ 400Hz) ---")

# Nomes dos Arquivos
ARQUIVO_DADOS = 'dados_processados_3_features_400Hz.npz'
ARQUIVO_MODELO_SAIDA = 'ann_MLP_3_features_400Hz.keras'
ARQUIVO_SCALER_SAIDA = 'scaler_3_features_400Hz.npz'  # Salva os parâmetros de normalização

# Carregamento e Preparação dos Dados
try:
    if not os.path.exists(ARQUIVO_DADOS):
        raise FileNotFoundError(f"Arquivo '{ARQUIVO_DADOS}' não encontrado.")
    
    # Carrega o arquivo .npz gerado na Etapa 1
    data = np.load(ARQUIVO_DADOS)
    X_train, y_train = data['X_train'], data['y_train']
    X_val, y_val = data['X_val'], data['y_val']
    X_test, y_test = data['X_test'], data['y_test']

    if X_train.size == 0:
        print("ERRO FATAL: Conjunto de treino está vazio. Não é possível treinar.")
        raise ValueError("Conjunto de treino vazio.")

    print(f"Dados de 3 caracteristicas @ 400Hz carregados de '{ARQUIVO_DADOS}'.")
    print(f"Shapes - Treino: {X_train.shape}, Validação: {X_val.shape}, Teste: {X_test.shape}")

except Exception as e:
    print(f"ERRO: {e}")
    raise

# Normalização (Z-SCORE)
# A rede neural será treinada com dados normalizados (média 0, desvio 1).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val) if X_val.size > 0 else np.array([]).reshape(0, scaler.mean_.shape[0])
X_test_scaled = scaler.transform(X_test) if X_test.size > 0 else np.array([]).reshape(0, scaler.mean_.shape[0])

print("\n--- Parâmetros de Normalização (Média e Desvio) ---")
print("Valores de 'media_features' (scaler.mean_):")
print(np.array2string(scaler.mean_, separator=', '))
print("\nValores de 'desvio_features' (scaler.scale_):")
print(np.array2string(scaler.scale_, separator=', '))
print("--------------------------------------------------\n")

# Salva a média e o desvio. Estes valores são cruciais para o código C++ do ESP32.
np.savez_compressed(ARQUIVO_SCALER_SAIDA, mean=scaler.mean_, scale=scaler.scale_)
print(f"Parâmetros de normalização salvos em: {ARQUIVO_SCALER_SAIDA}")

# Definição da Arquitetura da MLP (Rede Neural Multi-Layer Perceptron)
model_mlp = keras.Sequential([
    keras.Input(shape=(3,)),                      # Entrada: 3 neurônios (Area, SUT, Largura25)
    layers.Dense(35, activation='relu', name='Camada_Oculta_1'),
    layers.Dropout(0.3, name='Dropout_1'),         # Evita Overfitting
    layers.Dense(20, activation='relu', name='Camada_Oculta_2'),
    layers.Dropout(0.3, name='Dropout_2'),
    layers.Dense(2, name='Camada_Saida')           # Saída: 2 neurônios (SBP e DBP)
], name="MLP_3_Features")

model_mlp.summary()

# Compilação
model_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.Huber()  # Robusto contra outliers
)

# Callback para reduzir a taxa de aprendizado
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=10, 
    min_lr=0.0001, 
    verbose=1
)

# Treinamento
if X_train_scaled.shape[0] < 1 or X_val_scaled.shape[0] < 1:
    print("\nAVISO: Dados insuficientes. O treinamento será pulado.")
    history_mlp = None
else:
    print("\nIniciando treinamento da MLP...")
    history_mlp = model_mlp.fit(
        X_train_scaled, y_train,
        batch_size=64,
        epochs=100,
        validation_data=(X_val_scaled, y_val),
        callbacks=[reduce_lr],
        verbose=1
    )

# Avaliação Final
if history_mlp is not None and X_test_scaled.shape[0] > 0:
    print("\nAvaliando o modelo MLP no conjunto de teste...")
    previsoes = model_mlp.predict(X_test_scaled)
    
    diferencas = previsoes - y_test
    mae = np.mean(np.abs(diferencas), axis=0)
    sd_erro = np.std(diferencas, axis=0)

    print("\n--- Resultados FINAIS (MLP 3 Características @ 400Hz) ---")
    print(f"SBP (Sistólica) -> MAE: {mae[0]:.2f} mmHg | SD Erro: {sd_erro[0]:.2f} mmHg")
    print(f"DBP (Diastólica) -> MAE: {mae[1]:.2f} mmHg | SD Erro: {sd_erro[1]:.2f} mmHg")

    model_mlp.save(ARQUIVO_MODELO_SAIDA)
    print(f"\nModelo MLP salvo em: {ARQUIVO_MODELO_SAIDA}")

elif history_mlp is None:
    print("\nModelo não foi treinado por falta de dados.")
else:
    model_mlp.save(ARQUIVO_MODELO_SAIDA)
    print(f"\nTreinamento concluído. Modelo salvo: {ARQUIVO_MODELO_SAIDA}")

## ETAPA 3: Conversão para TensorFlow Lite

Para rodar o modelo no ESP32, convertemos o arquivo do Keras para o formato otimizado `.tflite`.

### Instruções:
1. Execute o script de conversão para gerar o arquivo binário em Float32.
2. Após gerar o arquivo `.tflite`, use o terminal (Git Bash) para transformá-lo em um array C++:

`xxd -i modelo_convertido.tflite > model_data.h`

### Finalização:
* Copie o arquivo `model_data.h` para a pasta do seu projeto no **PlatformIO**.
* Utilize os parâmetros de normalização(média e desvio) salvos no arquivo do Scaler para tratar os dados em tempo real antes da inferência.

In [ ]:
# SCRIPT DE CONVERSÃO TFLITE (FLOAT32)

import tensorflow as tf
import numpy as np
import os

# Nomes dos Arquivos
MODELO_KERAS_ENTRADA = "ann_MLP_3_features_400Hz.keras"
MODELO_TFLITE_SAIDA = "ann_MLP_3_features_400Hz_float32.tflite"

print("--- Iniciando Conversão para TensorFlow Lite ---")

if not os.path.exists(MODELO_KERAS_ENTRADA):
    raise FileNotFoundError(f"Arquivo do modelo '{MODELO_KERAS_ENTRADA}' não encontrado.")

# 1. Carrega o modelo treinado na Etapa 2
print(f"Carregando modelo Keras: {MODELO_KERAS_ENTRADA}")
model = tf.keras.models.load_model(MODELO_KERAS_ENTRADA)

# 2. Configura o conversor TFLite
# Usamos o formato Float32 para manter a precisão total no ESP32
converter = tf.lite.TFLiteConverter.from_keras_model(model)

print("Convertendo o modelo para TensorFlow Lite (Float32)...")
tflite_model_float = converter.convert()  # Executa a conversão
print("Conversão concluída.")

# 3. Salva o arquivo binário convertido
print(f"Salvando o modelo TFLite em: {MODELO_TFLITE_SAIDA}")
with open(MODELO_TFLITE_SAIDA, 'wb') as f:
    f.write(tflite_model_float)

print(f"\nSucesso! Modelo convertido para: {MODELO_TFLITE_SAIDA}")
print("-" * 50)
print("PRÓXIMO PASSO (No Terminal/Git Bash):")
print(f"Converta o arquivo .tflite em um array de bytes C++ (.h) usando o comando:")
print(f"xxd -i {MODELO_TFLITE_SAIDA} > model_data.h")
print("-" * 50)